# Symulacja układu ofiara–drapieżnik (Prey–Predator)

Notatnik zawiera kod przeprowadzonych eksperymentów (test powtarzalności, wykres historii populacji, histogram cechy `p_reproduce`, eksperyment bazowy oraz analizę wrażliwości parametrów) wraz z podsumowaniem wyników i wnioskami.

In [ ]:
import random
import statistics
from prey_predator_sim.species import Prey, Predator
from prey_predator_sim.population import Population

## 1. Test 100 kroków i powtarzalności (dla tego samego ziarna)

In [ ]:
random.seed(42)
pop_a = Population(
    {Prey: 30, Predator: 70},
    p_death=0.1,
    p_reproduce=0.2,
)
for _ in range(100):
    pop_a.natural_selection()

random.seed(42)
pop_b = Population(
    {Prey: 30, Predator: 70},
    p_death=0.1,
    p_reproduce=0.2,
)
for _ in range(100):
    pop_b.natural_selection()

print("Identyczne dla tego samego ziarna:", pop_a.history == pop_b.history)
print("Koniec (pop_a):", pop_a.n)
pop_a.plot_history()
pop_a.plot_histogram("p_reproduce")

## 2. Histogram dla żywej populacji (łagodniejsze parametry)

In [ ]:
pop_hist = Population({Prey: 50, Predator: 70}, p_death=0.05, p_reproduce=0.25)
for _ in range(150):
    pop_hist.natural_selection()

print("Koniec (pop_hist):", pop_hist.n)
pop_hist.plot_histogram("p_reproduce")

## 3. Eksperyment bazowy (10 powtórzeń, różne ziarna)

In [ ]:
def _std(vals):
    return statistics.stdev(vals) if len(vals) > 1 else 0.0

seeds = list(range(10))
end_prey = []
end_pred = []

for s in seeds:
    random.seed(s)
    pop = Population({Prey: 30, Predator: 70}, p_death=0.1, p_reproduce=0.2)
    for _ in range(100):
        pop.natural_selection()
    last = pop.history[-1]
    end_prey.append(last.get(Prey, 0))
    end_pred.append(last.get(Predator, 0))

prey_ext = sum(1 for x in end_prey if x == 0) / len(seeds) * 100
pred_ext = sum(1 for x in end_pred if x == 0) / len(seeds) * 100
prey_avg = sum(end_prey) / len(end_prey)
pred_avg = sum(end_pred) / len(end_pred)
prey_sd = _std(end_prey)
pred_sd = _std(end_pred)

print(f"Baza -> wymarcie Prey: {prey_ext:.1f}%, wymarcie Predator: {pred_ext:.1f}%")
print(
    f"Baza -> srednia koniec Prey: {prey_avg:.2f} (sd={prey_sd:.2f}), "
    f"Predator: {pred_avg:.2f} (sd={pred_sd:.2f})"
)

## 4. Analiza wrażliwości parametrów (`p_reproduce`, `p_death`)

In [ ]:
grid = {
    "p_reproduce": [0.1, 0.2, 0.3],
    "p_death": [0.05, 0.1, 0.2],
}

for param, vals in grid.items():
    for v in vals:
        p_prey = []
        p_pred = []
        for s in range(10):
            random.seed(100 + s)
            kw = {"p_death": 0.1, "p_reproduce": 0.2}
            kw[param] = v
            pop = Population({Prey: 30, Predator: 70}, **kw)
            for _ in range(100):
                pop.natural_selection()
            last = pop.history[-1]
            p_prey.append(last.get(Prey, 0))
            p_pred.append(last.get(Predator, 0))

        ext_prey = sum(1 for x in p_prey if x == 0) / 10 * 100
        ext_pred = sum(1 for x in p_pred if x == 0) / 10 * 100
        avg_prey = sum(p_prey) / 10
        avg_pred = sum(p_pred) / 10
        sd_prey = _std(p_prey)
        sd_pred = _std(p_pred)
        print(
            f"{param}={v} | ext Prey: {ext_prey:.0f}%, ext Pred: {ext_pred:.0f}% | "
            f"avg Prey: {avg_prey:.2f} (sd={sd_prey:.2f}), "
            f"avg Pred: {avg_pred:.2f} (sd={sd_pred:.2f})"
        )

## 5. Podsumowanie eksperymentów i wnioski

* **Przebieg symulacji i powtarzalność:** Ze względów wydajnościowych wykonałam eksperymenty na horyzoncie `100` kroków. Ustawiłam ziarno losowe (`random.seed`), co porządkuje losowanie w ramach danego przebiegu, ale między osobnymi uruchomieniami nadal pojawiają się różnice wynikające z losowego charakteru zdarzeń w modelu oraz iteracji po zbiorze (`set`) i alokacji obiektów w pamięci.
* **Dynamika populacji i wymieranie drapieżników:** W scenariuszu bazowym (`{Prey: 30, Predator: 70}`) oraz na moim wykresie historii populacji widać silną presję drapieżników na zasób ofiar na początku symulacji. Gdy ofiary spadają do zera, drapieżniki pozbawione pokarmu (przy uwzględnieniu śmiertelności oraz ryzyka śmierci z głodu) również stopniowo wymierają. Zaobserwowałam, że wymieranie drapieżników nie jest błędem implementacji, lecz naturalnym skutkiem przyjętych założeń modelu i parametrów.
* **Analiza wrażliwości (`p_reproduce`, `p_death`):** Uzyskane przeze mnie wyniki wykazują wyraźną nieliniowość i dużą wariancję między powtórzeniami. Przy niskim `p_reproduce = 0.1` oraz wyższych wartościach `p_death` (`0.1`, `0.2`) w 100% prób doszło do wymarcia obu gatunków (średnie końcowe `0.00`). Z kolei obniżenie śmiertelności do `p_death = 0.05` (wymarcie ofiar na poziomie `40%`, średnia końcowa `337357.70`) lub zwiększenie rozrodu do `p_reproduce = 0.3` (wymarcie ofiar `90%`, średnia końcowa `391721.10`) pokazało, że w pojedynczych przebiegach, w których garstka ofiar przetrwa po wyginięciu drapieżników, populacja ofiar może gwałtownie wzrosnąć.
* **Histogram i ograniczenia modelu:** Dla stanu końcowego po wymarciu populacji histogram wyszedł pusty z braku żywych osobników, a dla żywej populacji przy `p_reproduce` dał pojedynczy słupek na wartości początkowej, ponieważ model nie uwzględnia mutacji ani zróżnicowania dziedziczonego.

### Dlaczego moja symulacja nie jest w pełni powtarzalna?

* **Kolejność iteracji po `set`:** W Pythonie zbiór (`set`) nie ma ustalonej kolejności elementów, więc porządek przetwarzania osobników (i kolejność losowań) może zmieniać się między uruchomieniami.
* **Alokacja obiektów w pamięci:** Nowo tworzone obiekty trafiają pod różne adresy w pamięci, co wpływa na kolejność w strukturach takich jak `set`.
* **Stochastyczny charakter modelu:** Nawet przy tym samym ziarnie drobna zmiana kolejności interakcji między uruchomieniami może rozepchnąć dalszą trajektorię populacji.